# SGLang

[SGLang](https://github.com/sgl-project/sglang) is a fast serving framework for large language models. It provides an OpenAI-compatible HTTP server that LlamaIndex connects to via the `llama-index-llms-sglang` integration.

SGLang supports multiple hardware backends:
- **CUDA** (NVIDIA GPUs)
- **XPU** (Intel GPUs — Arc, Flex, Data Center GPU)
- **CPU** (fallback)

This notebook covers:
1. Installing dependencies and starting the SGLang server
2. Basic completion
3. Chat
4. Streaming
5. Async
6. Running on Intel XPU

## 1. Installation

In [ ]:
%pip install llama-index-llms-sglang

Note: you may need to restart the kernel to use updated packages.


## 2. Start the SGLang Server

Before running any of the cells below, start the SGLang server in a separate terminal.

**CUDA (NVIDIA GPU):**
```bash
pip install sglang[all]
python -m sglang.launch_server \
    --model-path mistralai/Mistral-7B-Instruct-v0.1 \
    --port 30000
```

**XPU (Intel GPU) via Docker:**

```bash
# Build the XPU image
docker build -t sglang-xpu:latest -f docker/xpu.Dockerfile .

# Launch the server
# If behind a corporate proxy, add -e http_proxy=... -e https_proxy=... to the docker run command
docker run \
  -it --rm --privileged \
  --ipc=host \
  --network=host \
  --user root \
  --group-add "$(getent group video | cut -d: -f3)" \
  --device /dev/dri \
  -v /dev/dri/by-path:/dev/dri/by-path \
  -v /dev/shm:/dev/shm \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -e ZE_AFFINITY_MASK=0,1 \
  -e ONEAPI_DEVICE_SELECTOR=level_zero:* \
  -e http_proxy="${http_proxy}" \
  -e https_proxy="${https_proxy}" \
  -e no_proxy="${no_proxy}" \
  -e ftp_proxy="${ftp_proxy}" \
  sglang-xpu:latest \
  /bin/bash -c 'sglang serve \
    --model-path Qwen/Qwen3-4B-Instruct-2507 \
    --trust-remote-code \
    --disable-overlap-schedule \
    --device xpu \
    --host 0.0.0.0 \
    --tp 2 \
    --attention-backend intel_xpu \
    --page-size 128 \
    --tool-call-parser qwen \
    --grammar-backend xgrammar'
```

Key flags:
- `--network=host`: exposes port 30000 directly on the host
- `ZE_AFFINITY_MASK=0,1`: selects XPU tile/device indices
- `ONEAPI_DEVICE_SELECTOR=level_zero:*`: forces Level Zero backend for Intel GPU
- `--tp 2`: tensor parallelism across 2 XPU devices
- `--attention-backend intel_xpu`: Intel-optimized attention kernel

Wait until you see `Server is ready` before proceeding.

## 3. Basic Setup

In [ ]:
from llama_index.llms.sglang import SGLang
from llama_index.core.llms import ChatMessage

llm = SGLang(
    model="Qwen/Qwen3-4B-Instruct-2507",
    api_url="http://localhost:30000",
    temperature=0.7,
    max_new_tokens=256,
    is_chat_model=True,
)

## 4. Completion

In [ ]:
response = llm.complete("What is a black hole?")
print(response)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## 5. Chat

In [ ]:
messages = [
    ChatMessage(role="system", content="You are a helpful assistant."),
    ChatMessage(
        role="user", content="Explain quantum entanglement in simple terms."
    ),
]
response = llm.chat(messages)
print(response)

## 6. Streaming

In [ ]:
# Streaming completion
for chunk in llm.stream_complete("Tell me a short story about a robot."):
    print(chunk.delta, end="", flush=True)

In [ ]:
# Streaming chat
messages = [ChatMessage(role="user", content="What is the speed of light?")]
for chunk in llm.stream_chat(messages):
    print(chunk.delta, end="", flush=True)

## 7. Async

In [ ]:
response = await llm.acomplete("What is the meaning of life?")
print(response)

In [ ]:
messages = [ChatMessage(role="user", content="What is the meaning of life?")]
response = await llm.achat(messages)
print(response)

In [ ]:
# Async streaming completion
async for chunk in await llm.astream_complete("Describe the solar system."):
    print(chunk.delta, end="", flush=True)

In [ ]:
# Async streaming chat
messages = [ChatMessage(role="user", content="Describe the solar system.")]
async for chunk in await llm.astream_chat(messages):
    print(chunk.delta, end="", flush=True)

## 8. Running on Intel XPU

SGLang supports Intel GPUs (Arc, Flex, Data Center GPU) via the `--device xpu` flag when launching the server. The LlamaIndex client requires no changes — it connects to the same HTTP endpoint regardless of the backend device.

To verify the server is using XPU, check the server startup logs for:
```
Device: xpu
```

Once the XPU server is running on port 30000, use the same `SGLang` client as above:

In [ ]:
# Same client code works regardless of whether server runs on CUDA or XPU
llm_xpu = SGLang(
    model="Qwen/Qwen3-4B-Instruct-2507",
    api_url="http://localhost:30000",
    temperature=0.7,
    max_new_tokens=256,
    is_chat_model=True,
)

response = llm_xpu.complete("What is a black hole?")
print(response)

## 9. LLM Metadata

In [ ]:
print(llm.metadata)